<a href="https://colab.research.google.com/github/pdelrc/170824/blob/main/MNIST_AE_ORIGINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoencoder on MNIST

This tutorial shows how to use Keras and TensorFlow to build and train a simple shallow AutoEncoder (AE) to generate samples from the MNIST dataset.

references:  
https://keras.io/examples/vision/mnist_convnet \
https://colab.research.google.com/github/trekhleb/machine-learning-experiments/blob/master/experiments/digits_recognition_mlp/digits_recognition_mlp.ipynb \
https://blog.keras.io/building-autoencoders-in-keras.html \
https://keras.io/api/layers/regularizers

aug 2022, hdaniel@ualg.pt \
Update March 10, 2025, jvo@ualg.pt \
sept 2025, autoencoder version, hdaniel@ualg.pt


## Step 1: Load the MNIST dataset and normalize

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from typing import Tuple

In [ ]:
# Model / data parameters
num_classes = 10
input_shape = (28, 28, 1)

# the data, split between train and test sets
# Note that the targets are not needed to train the auto encoder
(xTrain, yTrain), (xTest, _) = keras.datasets.mnist.load_data()

# Join train and test datasets
xTrain = np.concatenate([xTrain, xTest])

# Scale images to the [0, 1] range
xTrain = xTrain.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
xTrain = np.expand_dims(xTrain, -1)

#Display layout
print("xTrain shape:", xTrain.shape)
print(xTrain.shape[0], "train samples")

### Let's see part of the MNIST data set

In [ ]:
numbers_to_display = 25
num_cells = math.ceil(math.sqrt(numbers_to_display))
plt.figure(figsize=(10,10))
for i in range(numbers_to_display):
    plt.subplot(num_cells, num_cells, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(xTrain[i], cmap=plt.cm.binary)
    plt.xlabel(yTrain[i])
plt.show()

#
## Step 2: Create the AutoEncoder model

### Step 2.1: Define the AutoEncoder class

Class AutoEncoderMLP is a subclass of Keras.Model that defines an autoencoder with Dense layers, essentially 2 multi-layer perceptrons in a pipeline.

The method **_createEncoder()** defines the layers of the encoder, while the method **_createDecoder()** defines the layers of the encoder. These 2 methods accept a parameter named neurons that defines the number of neurons in the sole hidden layer. To add more layers, these parameter must be replaced by a list of integers: ```List[int]```.

The **call()** method joins the encoder and the decoder to form the autoencoder when calling the **fit()** and **predict()** methods.

In [ ]:
class AutoEncoderMLP(keras.Model):

    def __init__(self, inputShape:Tuple[int,int], encoderNeurons:int, decoderNeurons:int,
                 liReg:float, numLatentFeatures:int, **kwargs):
        super().__init__(**kwargs)
        self._encoder = self._createEncoder(inputShape, encoderNeurons, numLatentFeatures, l1Reg)
        self._decoder = self._createDecoder(inputShape, decoderNeurons, numLatentFeatures, l1Reg)

        # full autoencoder is accessed by methods fit() and predict()
        # because it is defined in method: call(self)

    def encoder(self):     return self._encoder
    def decoder(self):     return self._decoder

    def _createEncoder(self, inputShape:Tuple[int,int], neurons:int, numlatentFeatures:int, l1Reg:int):
        '''Define the encoder'''
        encoderInput = layers.Input(shape=inputShape)
        flat = layers.Flatten()(encoderInput)
        h = layers.Dense(neurons, activation ='relu', activity_regularizer=keras.regularizers.l1(l1Reg))(flat)
        encoderOutput = keras.layers.Dense(numlatentFeatures, activation='relu')(h)
        return keras.Model(inputs=encoderInput, outputs=encoderOutput, name="MLPencoder")


    def _createDecoder(self, inputShape:Tuple[int,int], neurons:int, numlatentFeatures:int, l1Reg:int):
        '''Define the decoder'''
        decoderInput = keras.Input(shape=(numlatentFeatures,))
        h = layers.Dense(neurons, activation ='relu', activity_regularizer=keras.regularizers.l1(l1Reg))(decoderInput)
        flat = keras.layers.Dense(inputShape[0]*inputShape[1], activation='sigmoid')(h)
        decoderOutput = layers.Reshape(inputShape)(flat)
        return keras.Model(inputs=decoderInput, outputs=decoderOutput, name="MPLdecoder")


    #Subclasses of keras.model
    #need to define forward pass to fit, predict, ... methods
    #
    #optionally can have a boolean argument: training
    #which can be used to specify a different behavior in training and inference,
    #see examples in:
    #https://keras.io/api/models/model/
    def call(self, data, training=False):
        encoderOutput = self._encoder(data)
        return self._decoder(encoderOutput)


### Step 2.2: Define the AutoEncoder

In [ ]:
digitShape = xTrain[0].shape
neurons = 128
numLatentFeatures = 2
l1Reg = 10e-5

ae = AutoEncoderMLP((28,28), neurons, neurons, l1Reg, numLatentFeatures)

### Step 2.3: The encoder architecture

In [ ]:
encoder = ae.encoder()
encoder.summary()

###
### Step 2.3: The decoder architecture

In [ ]:
encoder = ae.decoder()
encoder.summary()

#
## Step 3: Train the  model on the MNIST dataset

In [ ]:
batchSize = 128
epochs    = 20

# Set optimiser to Adam and loss to Mean Squared Error
ae.compile(optimizer='adam', loss='mse')

# Fit(X,y), both X and y are the same xTrain, since it is an autoencoder
# it is intendent to learn how to encode and decode xTrain digits
trainingHistory = ae.fit(xTrain, xTrain, batch_size=batchSize, epochs=epochs, shuffle=True, validation_split=0.1)

## Step 4: Evaluate the model on the MNIST dataset

### Loss function evolution during the training

In [ ]:
plt.xlabel('Epoch Number')
plt.ylabel('Loss')
plt.plot(trainingHistory.history['loss'], label='training set')
plt.plot(trainingHistory.history['val_loss'], label='test set')
plt.legend()

## Step 5: Synthesize new images from the learned latent space

In [ ]:
numGenerate = 100

# Sample white noise random vectors from a normal distribution
zSample = np.random.normal(size=(numGenerate, numLatentFeatures))

# Decode the random noisy vectors into images
generated = ae.decoder().predict(zSample)

## Step 6: Plot the synthesized images

In [ ]:
plt.figure(figsize=(10,10))
for i in range(numGenerate):
    plt.subplot(10, 10, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(generated[i], cmap=plt.cm.binary)
plt.show()